# ABDS カード画像ダウンローダー

Arsenal Base Deck Simulator 用のカード画像をGoogleドライブにダウンロードします。

## 使い方
1. 上のメニューから「ランタイム」→「すべてのセルを実行」
2. Googleドライブへのアクセス許可を求められたら「許可」
3. 完了まで待つ（約15〜30分）
4. Googleドライブの `ABDS_CardImages/` フォルダに画像が保存されます
5. ABDSアプリの「キャッシュ管理」→「画像インポート」でフォルダを選択

---
**注意**: カード画像の著作権はBANDAI/創通・サンライズ等の権利者に帰属します。画像の取得・使用はユーザーの自己責任で行ってください。第三者への再配布は禁止です。

## Step 1: Googleドライブをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: 設定

In [ ]:
import os
import json
import urllib.request
import time
from IPython.display import clear_output

# === 設定 ===
# ダウンロード先 (Googleドライブ内のフォルダ)
OUTPUT_DIR = '/content/drive/MyDrive/ABDS_CardImages'

# カード番号リスト取得元
CARD_LIST_URL = 'https://sekimiya.github.io/abds/data/card_numbers.json'

# 画像ベースURL
IMG_BASE = 'https://www.gundam-ab.com/images/cardlist/card/'

# リクエストヘッダー
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'https://www.gundam-ab.com/cardlist/'
}

# 同時リクエスト間隔 (秒) - サーバー負荷軽減
DELAY = 0.05

# True: 既にDL済みの画像はスキップ / False: 全て再DL
SKIP_EXISTING = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'保存先: {OUTPUT_DIR}')
print('設定OK')

## Step 3: カード番号リストを取得

In [ ]:
req = urllib.request.Request(CARD_LIST_URL)
with urllib.request.urlopen(req, timeout=30) as resp:
    card_numbers = json.loads(resp.read().decode('utf-8'))

print(f'カード数: {len(card_numbers)}')
print(f'画像数 (表+裏): {len(card_numbers) * 2}')
print(f'先頭5枚: {card_numbers[:5]}')
print(f'末尾5枚: {card_numbers[-5:]}')

## Step 4: ダウンロード実行

全カードの表面・裏面画像をダウンロードします。途中で止めても、再実行すれば続きから再開されます。

In [ ]:
def download_image(card_num, suffix=''):
    """1枚の画像をダウンロード"""
    fname = f'{card_num}{suffix}.jpg'
    fpath = os.path.join(OUTPUT_DIR, fname)

    if SKIP_EXISTING and os.path.exists(fpath) and os.path.getsize(fpath) > 1000:
        return 'skip'

    url = f'{IMG_BASE}{card_num}{suffix}.jpg'
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = resp.read()
            if len(data) > 1000:
                with open(fpath, 'wb') as f:
                    f.write(data)
                return 'ok'
            else:
                return 'fail'
    except Exception as e:
        return 'fail'

# --- メインループ ---
total = len(card_numbers) * 2
done = 0
downloaded = 0
skipped = 0
failed = 0
start_time = time.time()

print(f'ダウンロード開始: {len(card_numbers)}枚 x 2(表裏) = {total}枚')
print('=' * 50)

for i, card_num in enumerate(card_numbers):
    for suffix in ['', '_b']:
        result = download_image(card_num, suffix)
        if result == 'ok':
            downloaded += 1
        elif result == 'skip':
            skipped += 1
        else:
            failed += 1
        done += 1

        if DELAY > 0 and result == 'ok':
            time.sleep(DELAY)

    # 進捗表示 (50枚ごと)
    if (i + 1) % 50 == 0 or i == len(card_numbers) - 1:
        elapsed = time.time() - start_time
        rate = done / elapsed if elapsed > 0 else 0
        remaining = (total - done) / rate if rate > 0 else 0
        clear_output(wait=True)
        pct = done / total * 100
        bar = '#' * int(pct // 2) + '-' * (50 - int(pct // 2))
        print(f'[{bar}] {pct:.1f}%')
        print(f'{done}/{total} (DL:{downloaded} skip:{skipped} fail:{failed})')
        print(f'経過: {elapsed:.0f}秒 / 残り: {remaining:.0f}秒')
        print(f'最終: {card_num}')

# --- 完了 ---
elapsed = time.time() - start_time
clear_output(wait=True)
print('=' * 50)
print(f'ダウンロード完了!')
print(f'  DL成功: {downloaded}枚')
print(f'  スキップ(DL済み): {skipped}枚')
print(f'  失敗: {failed}枚')
print(f'  所要時間: {elapsed:.0f}秒 ({elapsed/60:.1f}分)')
print(f'  保存先: {OUTPUT_DIR}')
print('=' * 50)
if failed > 0:
    print(f'\n{failed}枚の失敗があります。再実行すると失敗分のみリトライします。')
print(f'\nGoogleドライブの「ABDS_CardImages」フォルダを確認してください。')
print('ABDSアプリの「キャッシュ管理」→「画像インポート」でフォルダを選択すれば完了です。')

## (オプション) 特定シリーズのみダウンロード

全カードではなく特定シリーズだけDLしたい場合はこちらを使ってください。

In [ ]:
# DLしたいシリーズのプレフィックスを指定
# 例: 'BP07' / 'VEB01' / 'PR-' / 'VE01'
TARGET_SERIES = 'BP07'

filtered = [n for n in card_numbers if n.startswith(TARGET_SERIES)]
print(f'{TARGET_SERIES}: {len(filtered)}枚')

for i, card_num in enumerate(filtered):
    for suffix in ['', '_b']:
        result = download_image(card_num, suffix)
        status = 'DL' if result == 'ok' else 'skip' if result == 'skip' else 'FAIL'
        print(f'  [{status}] {card_num}{suffix}.jpg')
        if DELAY > 0 and result == 'ok':
            time.sleep(DELAY)

print(f'\n{TARGET_SERIES} 完了')